# Tool-trajectory evaluation before agent release

This credential-free lab adapts MLflow's [LangGraph agent cookbook](https://mlflow.org/cookbook/langgraph-agent/) to the platform contract. A fluent final answer is not enough: an agent can quote the right fact while calling an unapproved tool, using the wrong arguments, or skipping a required lookup.

The lab starts with deterministic exact matching because release gates need reproducible evidence. A connected project may add MLflow's fuzzy `ToolCallCorrectness` judge as report-only evidence after routing it through the governed judge model.

The repository's executable LangGraph implementation remains the optional native recipe under `templates/agent-app/template/recipes/langgraph/`. It owns graph state, durable checkpoints, interrupts, and idempotency; `aai-core` does not wrap those APIs.

## 1. Define final-answer and trajectory expectations

Each row keeps `expected_facts` separate from `expected_tool_calls`. The second case is intentional: its answer contains the right number, but it came from a cached-summary tool instead of the governed source lookup. Output-only evaluation would miss that failure.

In [ ]:
EVAL_CASES = [
    {
        "case_id": "revenue-from-governed-source",
        "inputs": {"question": "What was fictional quarterly revenue?"},
        "expectations": {
            "expected_facts": ["$128.4 million", "ARS-FY25-Q2-RESULTS"],
            "expected_tool_calls": [
                {
                    "name": "lookup_earnings_source",
                    "arguments": {"source_id": "ARS-FY25-Q2-RESULTS"},
                }
            ],
        },
        "observed": {
            "answer": "$128.4 million [source: ARS-FY25-Q2-RESULTS]",
            "tool_calls": [
                {
                    "name": "lookup_earnings_source",
                    "arguments": {"source_id": "ARS-FY25-Q2-RESULTS"},
                }
            ],
        },
    },
    {
        "case_id": "right-answer-wrong-trajectory",
        "inputs": {"question": "What was fictional free cash flow?"},
        "expectations": {
            "expected_facts": ["$21.7 million", "ARS-FY25-Q2-CASH-RISK"],
            "expected_tool_calls": [
                {
                    "name": "lookup_earnings_source",
                    "arguments": {"source_id": "ARS-FY25-Q2-CASH-RISK"},
                }
            ],
        },
        "observed": {
            "answer": "$21.7 million [source: ARS-FY25-Q2-CASH-RISK]",
            "tool_calls": [
                {
                    "name": "lookup_cached_summary",
                    "arguments": {"issuer": "aster-ridge-systems"},
                }
            ],
        },
    },
]

## 2. Compare calls as a multiset of exact name/argument signatures

A multiset preserves duplicate calls, so calling the same expensive tool twice cannot collapse into one passing set member. Ordering is ignored here because these lookups are independent; make ordering explicit when tool sequence is part of the contract.

In [ ]:
import json
from collections import Counter

import pandas as pd


def call_signature(call):
    return (
        str(call["name"]),
        json.dumps(
            call.get("arguments", {}),
            ensure_ascii=True,
            separators=(",", ":"),
            sort_keys=True,
        ),
    )


def score_case(case):
    expected = Counter(
        call_signature(call)
        for call in case["expectations"]["expected_tool_calls"]
    )
    observed = Counter(
        call_signature(call) for call in case["observed"]["tool_calls"]
    )
    answer = case["observed"]["answer"]
    expected_facts = case["expectations"]["expected_facts"]
    return {
        "case_id": case["case_id"],
        "final_answer_correct": all(fact in answer for fact in expected_facts),
        "tool_trajectory_exact": observed == expected,
        "missing_calls": list((expected - observed).elements()),
        "unexpected_calls": list((observed - expected).elements()),
    }


trajectory_report = pd.DataFrame(score_case(case) for case in EVAL_CASES)
trajectory_report

In [ ]:
wrong_path = trajectory_report.set_index("case_id").loc[
    "right-answer-wrong-trajectory"
]
assert bool(wrong_path["final_answer_correct"])
assert not bool(wrong_path["tool_trajectory_exact"])

gate_passed = bool(
    trajectory_report["final_answer_correct"].all()
    and trajectory_report["tool_trajectory_exact"].all()
)
decision = "adopt" if gate_passed else "reject"
{
    "measurement_source": "simulated_offline_fixture",
    "gate_passed": gate_passed,
    "decision": decision,
    "release": "blocked" if not gate_passed else "eligible",
}

## 3. Carry the contract into the connected agent template

In a generated `agent-app`, use `app.tool_scoring.exact_tool_call_scorer()` in the release gate. The native MLflow evaluation rows use the same `inputs`, `expected_facts`, and `expected_tool_calls` shape shown above.

For the optional LangGraph recipe:

1. Install its certified lock and inject an async durable checkpointer and persistent store.
2. Configure only `TraceIntegration.MLFLOW_LANGCHAIN` with `run_tracer_inline=True`; do not add SDK provider spans around the same graph call.
3. Give every `ainvoke()` or resume its own trace context while reusing the opaque checkpoint/session ID.
4. Keep interrupts before side effects and protect resumed execution with an idempotency key.
5. Evaluate traced calls with the deterministic exact scorer. If fuzzy `ToolCallCorrectness` is also useful, route it through the configured judge model and keep it report-only until calibrated.

This lab rejects the observed change because one critical trajectory failed, even though aggregate final-answer correctness is 100%.

The optional cell below persists this synthetic contract as a Unity Catalog EvaluationDataset and records a described MLflow result run. It creates no agent calls, prompts, or traces.

In [ ]:
PERSIST_EVIDENCE_TO_DATABRICKS = False

if PERSIST_EVIDENCE_TO_DATABRICKS:
    import mlflow

    from aai_core.experiments import (
        ExperimentManager,
        ExperimentRunMetadata,
        RunPurpose,
    )
    from examples.notebook_setup import (
        get_or_create_uc_evaluation_dataset,
        preflight_databricks_evidence,
        prepare_notebook_environment,
    )

    environment = prepare_notebook_environment(
        evidence_destination="databricks"
    )
    evidence = preflight_databricks_evidence(environment)
    dataset = get_or_create_uc_evaluation_dataset(
        evidence=evidence,
        dataset_name="fictional_agent_tool_trajectory_regression_v1",
        records=[
            {
                "inputs": case["inputs"],
                "expectations": case["expectations"],
                "outputs": case["observed"],
            }
            for case in EVAL_CASES
        ],
        mlflow_module=mlflow,
    )
    experiments = ExperimentManager(
        experiment_name=evidence.experiment_name,
        context=evidence.context.tags,
    )
    with experiments.run(
        run_name="tool-trajectory-simulated-result",
        description=(
            "Simulated deterministic tool-trajectory result for the governed "
            "fictional agent regression dataset; no model was invoked."
        ),
        parameters={"measurement_source": "simulated_offline_fixture"},
        metadata=ExperimentRunMetadata(
            purpose=RunPurpose.RESULT,
            change_id="tool-trajectory-contract-v1",
            change_summary="Require exact governed tool names and arguments.",
        ),
    ) as evidence_run:
        mlflow.log_input(dataset, context="tool_trajectory_evaluation")
        mlflow.log_metrics(
            {
                "final_answer_pass_rate": float(
                    trajectory_report["final_answer_correct"].mean()
                ),
                "exact_trajectory_pass_rate": float(
                    trajectory_report["tool_trajectory_exact"].mean()
                ),
            }
        )
        mlflow.log_table(
            trajectory_report.to_dict(orient="records"),
            artifact_file="evaluation/tool_trajectory_report.json",
        )
        print(
            {
                "run_id": evidence_run.info.run_id,
                "dataset": dataset.name,
                "dataset_id": dataset.dataset_id,
            }
        )
else:
    print("DATABRICKS EVIDENCE PERSISTENCE SKIPPED")